# Module 4 – Vector Databases and Pinecone Integration
### Course: OpenAI Embeddings API | Pluralsight
---
## What You'll Learn
- Why vector databases are needed beyond in-memory numpy
- Vector DB landscape: Pinecone, Weaviate, Qdrant, Milvus, Chroma, FAISS
- Setting up a Pinecone serverless index
- Upserting embeddings with metadata and querying with filters
- Production considerations: updates, deletes, cost optimization
---

In [ ]:
# WHY NUMPY BREAKS AT SCALE
print("Memory required for embedding matrix (float32):")
print(f"{'Reviews':>12} {'Dims':>6} {'Memory (GB)':>14}")
print("-" * 35)
for n_reviews in [500, 50_000, 1_000_000, 50_000_000]:
    mem_gb = (n_reviews * 1536 * 4) / (1024**3)
    print(f"{n_reviews:>12,} {1536:>6} {mem_gb:>14.2f}")

In [ ]:
%pip install -q openai pinecone

---
## Clip 2: Hands-on — OpenAI Embeddings with Pinecone

**SDK note:** Pinecone v5+ uses `CloudProvider` and `AwsRegion` enums for `ServerlessSpec`.
Use `pc.Index(host=index_config.host)` to get an index handle (preferred over `pc.Index(name)`).

Ref: [sdk.pinecone.io/python/rest.html](https://sdk.pinecone.io/python/rest.html)

In [ ]:
# - 2.1 Create Pinecone index

import os
import getpass
import json
import time
import random
from openai import OpenAI

# Latest Pinecone SDK (v5+) uses enum-based cloud/region spec
# Ref: https://sdk.pinecone.io/python/rest.html
from pinecone import Pinecone, ServerlessSpec, CloudProvider, AwsRegion

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Initialize Pinecone with your API key
# Free account: https://app.pinecone.io
os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter Pinecone API key: ")
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# Load embedded reviews from Module 2
with open("../module2/yelp_embeddings.json") as f:
    embedded_reviews = json.load(f)

print(f"Loaded {len(embedded_reviews)} embedded reviews")
print(f"OpenAI client ready: {openai_client is not None}")
print(f"Pinecone client ready: {pc is not None}")

In [ ]:
# - 2.2 Create Pinecone index
INDEX_NAME = "yelp-restaurant-finder"
DIMENSION = 1536   # text-embedding-3-small default
METRIC = "cosine"

# Create index if it doesn't already exist
existing_indexes = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    print(f"Creating index '{INDEX_NAME}'...")

    ## TODO: #1 - Create the index with the specified name, dimension, and metric.
    
    
    # Wait for index to be ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(2)
    print("Index ready!")
else:
    print(f"Index '{INDEX_NAME}' already exists.")
    index_config = pc.describe_index(INDEX_NAME)

# Get index handle via host (recommended in v5+)
index = pc.Index(host=index_config.host)
print(f"\nIndex stats: {index.describe_index_stats()}")

In [ ]:
# - 2.3 Upsert embeddings with metadata

# UPSERT EMBEDDINGS WITH METADATA
UPSERT_BATCH_SIZE = 100
NAMESPACE = "yelp-reviews"

random.seed(42)
cities = ["New York", "Los Angeles", "Chicago", "San Francisco", "Austin"]
cuisines = ["American", "Italian", "Mexican", "Japanese", "Chinese", "Indian", "Mediterranean"]

# Pinecone vector record format:
# {"id": str, "values": list[float], "metadata": dict}
# OR tuple: (id_str, values_list, metadata_dict)
records = [
    {
        "id": str(r["id"]),
        "values": r["embedding"],
        "metadata": {
            "stars":   r["stars"],
            "text":    r["text"][:200],
            "city":    random.choice(cities),
            "cuisine": random.choice(cuisines),
        }
    }
    for r in embedded_reviews
]

print(f"Upserting {len(records)} vectors in batches of {UPSERT_BATCH_SIZE}...")

start = time.time()

for i in range(0, len(records), UPSERT_BATCH_SIZE):
    batch = records[i:i + UPSERT_BATCH_SIZE]

    ## TODO: #2 - Upsert the batch to the index with the specified namespace.
    
    if (i // UPSERT_BATCH_SIZE + 1) % 5 == 0 or i + UPSERT_BATCH_SIZE >= len(records):
        print(f"  Upserted {min(i + UPSERT_BATCH_SIZE, len(records))}/{len(records)} vectors")

print(f"\nDone in {time.time() - start:.1f}s")
print(f"Index stats: {index.describe_index_stats()}")

In [ ]:
# - 2.4 Search Pinecone with embedded query

def search_pinecone(
    query: str,
    top_k: int = 5,
    filter: dict = None,
    namespace: str = NAMESPACE
) -> list[dict]:
    """Embed a query and search Pinecone."""
    # Step 1: Generate query embedding
    # Ref: https://platform.openai.com/docs/api-reference/embeddings/create
    response = openai_client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    )
    query_vec = response.data[0].embedding

    # Step 2: Query Pinecone
    
    ## TODO: #3 - Query the index

    return [
        {
            "score":   match.score,
            "id":      match.id,
            "stars":   match.metadata.get("stars"),
            "city":    match.metadata.get("city"),
            "cuisine": match.metadata.get("cuisine"),
            "text":    match.metadata.get("text"),
        }
        for match in results.matches
    ]


# Basic search
query = "cozy romantic dinner with great wine"
print(f"Query: '{query}'\n")
for r in search_pinecone(query, top_k=3):
    print(f"  [score={r['score']:.4f}, {r['stars']}★, {r['city']}, {r['cuisine']}]")
    print(f"    {r['text'][:90]}...")

# Filtered search: Italian, 4+ stars
print("\n--- Filtered: Italian cuisine, 4+ stars ---")
for r in search_pinecone(
    query,
    top_k=3,
    ## TODO: #4 -  Add a filter for Italian cuisine and 4+ stars
):
    print(f"  [score={r['score']:.4f}, {r['stars']}★, {r['city']}, {r['cuisine']}]")
    print(f"    {r['text'][:90]}...")

In [ ]:
# - 2.5 Namespaces

# NAMESPACE DEMO: per-city partitioning

# Upsert a small subset into city-specific namespaces
nyc_records = [r for r in records if r["metadata"]["city"] == "New York"][:10]
la_records  = [r for r in records if r["metadata"]["city"] == "Los Angeles"][:10]

## TODO: #5 - Upsert the records to their respective city namespaces


print("Upserted city-specific namespaces")

# Query a specific city namespace
print("\nSearch in 'city:new-york' namespace only:")

## TODO: #6 - Query the "city:new-york" namespace for the same query and print results


In [ ]:
# - 2.6 Update and Delete

# UPDATE: upsert with same id overwrites the vector + metadata
updated_record = {
    "id": "0",
    "values": openai_client.embeddings.create(
        input="UPDATED: Management improved. Now a top dining destination!",
        model="text-embedding-3-small"
    ).data[0].embedding,
    "metadata": {"stars": 5, "text": "UPDATED: Management improved.", "city": "New York", "cuisine": "Italian"}
}

## TODO: #7 - Upsert the updated record to overwrite the existing vector with id '0'


print("Updated vector id='0'")

# Verify update
fetched = index.fetch(ids=["0"], namespace=NAMESPACE)
print(f"Fetched id='0': stars={fetched.vectors['0'].metadata['stars']}, text={fetched.vectors['0'].metadata['text'][:50]}")

# DELETE: remove vectors by id
index.delete(ids=["1", "2"], namespace=NAMESPACE)
print("\nDeleted ids '1' and '2'")

# Verify delete
fetch_deleted = index.fetch(ids=["1", "2"], namespace=NAMESPACE)
print(f"After delete, fetched vectors: {list(fetch_deleted.vectors.keys())}  (empty = deleted)")